# Wireless Networking and Mobile Computing

### Prerequisites

This Jupyter Notebook has been tested with Visual Studio Code, running in a local Python environment.

**Installs**

In [ ]:
# --- Installs: ensure required packages are available ---
%pip install numpy --quiet

**Imports**

In [ ]:
# --- Imports: libraries used in this notebook ---
import numpy as np

**Declarations**

Set the variables below and run the cell.

- `data_bits` defines the four-bit information block to encode
- `flip_index` selects a bit position to flip (0-based) for testing error correction
    - set flip_index = None to disable bit corruption

In [ ]:
# --- Declarations: editable variables for students ---
data_bits = ...
flip_index = ...

### 1. Data Redundancy

#### 1.1 Forward Error Correction (FEC)

Forward error correction adds redundancy to data so a receiver can detect and correct some errors without requiring retransmission.
This is useful in wireless environments where acknowledgments may be delayed or impossible (broadcast, multicast, UDP, high mobility links).

The transmitter applies a coding rule that adds structured redundancy. The receiver uses the same rule to identify and correct errors based on parity relationships. The choice of code and the amount of redundancy determine how many bit errors can be tolerated.

Typical systems add between ten and fifty percent redundancy depending on channel quality and target reliability. Modern wireless PHY layers combine multiple coding mechanisms, including LDPC and polar codes in 5G systems, and LDPC codes in Wi-Fi 6.

Two common coding families

| Code type           | Description                                   | Notes                                         |
| ------------------- | --------------------------------------------- | --------------------------------------------- |
| Block codes         | operate on fixed-size bit blocks              | examples: Hamming, Reed–Solomon               |
| Convolutional codes | operate on continuous bit streams with memory | typically decoded using the Viterbi algorithm |

Block-based and convolutional codes are often combined in layered systems.


#### 1.2. Hamming(8,4) Code
The Hamming(8,4) code maps four data bits into an eight-bit codeword. The added parity bits allow single-bit error correction and double-bit error detection.

In [ ]:
# --- Hamming(8,4) Demo ---

def encode_hamming_8_4(data):
    d = np.array(data) & 1
    cw = np.zeros(8, dtype=int)

    # place data bits at positions 3,5,6,7 (0-indexed: 2,4,5,6)
    cw[2], cw[4], cw[5], cw[6] = d

    # parity bits for Hamming positions 1,2,4 plus overall parity
    cw[0] = cw[2] ^ cw[4] ^ cw[6]
    cw[1] = cw[2] ^ cw[5] ^ cw[6]
    cw[3] = cw[4] ^ cw[5] ^ cw[6]
    cw[7] = cw.sum() % 2

    return cw

def decode_hamming_8_4(cw):
    cw = np.array(cw) & 1

    # parity checks
    s1 = cw[0] ^ cw[2] ^ cw[4] ^ cw[6]
    s2 = cw[1] ^ cw[2] ^ cw[5] ^ cw[6]
    s3 = cw[3] ^ cw[4] ^ cw[5] ^ cw[6]
    syndrome = s1*1 + s2*2 + s3*4
    overall = cw.sum() % 2

    # single-bit error correction
    if syndrome != 0 and overall == 1:
        cw[syndrome-1] ^= 1

    # extract data bits
    return cw[[2,4,5,6]]

# encode
codeword = encode_hamming_8_4(data_bits)

# simulate a single bit error (optional)
if flip_index is not None:
    codeword_noisy = codeword.copy()
    codeword_noisy[flip_index] ^= 1
else:
    codeword_noisy = codeword.copy()

# decode
decoded_bits = decode_hamming_8_4(codeword_noisy)

print("Original:", data_bits)
print("Encoded:", codeword.tolist())
print("Received:", codeword_noisy.tolist())
print("Decoded:", decoded_bits.tolist())